# Notebook 3 of 5: BiLSTM trained from scratch
### Swahili Tweet Sentiment Classification (AfriSenti)

**Before running:** pick the `.venv` kernel (top right). A CUDA GPU is used if there is one; a CPU also works here because the data is small, just slower.

This notebook:
1. Sets up the project folder (fixed split)
2. Builds a vocabulary from the training tweets only
3. Trains a bidirectional LSTM with early stopping on **validation macro-F1**
4. Repeats the whole training with **5 random seeds**, because with only 1,810 training tweets one run can be lucky or unlucky
5. Reports the mean and spread over seeds, evaluates each run on the test set once, and saves results for the best-validation run

**Same input as the baseline:** the cleaned tweet text. Only the model changes.

## Step 0 - Set up the project folder

Pick the `.venv` kernel (top right of the notebook). The libraries come from `requirements.txt` (`pip install -r requirements.txt` in the `.venv`).

The next cell moves to the project root. Everything this notebook writes goes inside the project folder, so there is nothing to clone or upload.

In [ ]:
from getpass import getpass
from pathlib import Path
import os

# VS Code starts a notebook inside its own folder (notebooks/). Move to the project root so the
# relative paths used below (data/, figures/, results/ ...) resolve from there.
ROOT = Path.cwd().resolve()
while not (ROOT / "notebooks").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "notebooks").is_dir(), "Open the project folder in VS Code and run this notebook from inside it."
os.chdir(ROOT)
print("Project folder:", ROOT)

for folder in ["results/confusion_matrices", "results/metrics", "results/errors", "results/predictions", "figures/bilstm", "models"]:
    os.makedirs(folder, exist_ok=True)

## Step 1 - GPU check and load the fixed split

In [ ]:
import torch, random, time, numpy as np, pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

LABELS = ["negative", "neutral", "positive"]
N_CLASSES = 3
train = pd.read_csv("data/processed/train.csv")
val = pd.read_csv("data/processed/val.csv")
test = pd.read_csv("data/processed/test.csv")
for df in (train, val, test):
    df["text_clean"] = df["text_clean"].fillna("").astype(str)
print("train:", len(train), " val:", len(val), " test:", len(test))

## Step 2 - Vocabulary and sequence length

The vocabulary comes from the **training set only**. Words seen fewer than 2 times become `<UNK>`. Swahili tweets have many one-off word forms (notebook 1 showed that), so expect a high `<UNK>` share. That is one reason a subword model is expected to do better in notebook 4.

Tweets are short, so sequences are cut at the 99th percentile length.

In [ ]:
from collections import Counter

PAD, UNK = 0, 1
counter = Counter(w for t in train["text_clean"] for w in t.split())
words = [w for w, c in counter.most_common() if c >= 2]
word2id = {w: i + 2 for i, w in enumerate(words)}
VOCAB_SIZE = len(word2id) + 2
print("Vocabulary size:", VOCAB_SIZE, "(of", len(counter), "distinct training words)")

lens = train["text_clean"].str.split().str.len()
MAX_LEN = int(np.ceil(lens.quantile(0.99)))
print("MAX_LEN (99th percentile):", MAX_LEN)

def encode(texts):
    ids = np.zeros((len(texts), MAX_LEN), dtype=np.int64)
    lengths = np.zeros(len(texts), dtype=np.int64)
    for i, t in enumerate(texts):
        toks = [word2id.get(w, UNK) for w in t.split()[:MAX_LEN]] or [UNK]
        ids[i, :len(toks)] = toks
        lengths[i] = len(toks)
    return torch.tensor(ids), torch.tensor(lengths)

Xtr, Ltr = encode(train["text_clean"].tolist()); ytr = torch.tensor(train["label_id"].values)
Xva, Lva = encode(val["text_clean"].tolist());   yva = torch.tensor(val["label_id"].values)
Xte, Lte = encode(test["text_clean"].tolist());  yte = torch.tensor(test["label_id"].values)

unk_rate = (Xtr == UNK).sum().item() / (Xtr != PAD).sum().item()
unk_val = (Xva == UNK).sum().item() / (Xva != PAD).sum().item()
print(f"Share of <UNK> tokens: train {unk_rate:.1%}, validation {unk_val:.1%}")

## Step 3 - Model

Embedding (100 numbers per word, learned from scratch) feeds a bidirectional LSTM (64 units per direction). The outputs are summarised by mean pooling and max pooling over the tweet, joined, passed through dropout and a final layer with 3 outputs. Padding is skipped with `pack_padded_sequence`. Class weights in the loss counter the imbalance, as `class_weight="balanced"` did for the baseline.

The model is small on purpose: with 1,810 tweets a large network would only memorise them.

In [ ]:
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim=100, hidden=64, n_classes=3, dropout=0.5):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=PAD)
        self.lstm = nn.LSTM(emb_dim, hidden, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden * 4, n_classes)

    def forward(self, x, lengths):
        e = self.drop(self.emb(x))
        packed = pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=x.size(1))
        mask = (torch.arange(x.size(1), device=x.device)[None, :] < lengths[:, None]).unsqueeze(-1)
        mean = (out * mask).sum(1) / lengths[:, None].float()
        mx = out.masked_fill(~mask, -1e9).max(1).values
        return self.fc(self.drop(torch.cat([mean, mx], dim=1)))

counts = np.bincount(train["label_id"].values, minlength=N_CLASSES)
class_w = torch.tensor(len(train) / (N_CLASSES * counts), dtype=torch.float).to(device)
print("Class weights:", dict(zip(LABELS, class_w.cpu().numpy().round(2))))
print("Parameters per model:", f"{sum(p.numel() for p in BiLSTMClassifier(VOCAB_SIZE).parameters()):,}")

## Step 4 - Train with early stopping, for several seeds

For each seed the weights start differently and the data is shuffled differently. After every epoch the model is scored on the validation set; the best epoch's weights are kept, and training stops after 5 epochs without improvement. The best weights are then scored on the **test** set once. Which run is "the representative run" for the confusion matrix and error list is chosen by **validation** score, never by test score.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score

SEEDS = [1, 2, 3, 4, 5]
BATCH, EPOCHS, PATIENCE, LR = 32, 40, 5, 1e-3
criterion = nn.CrossEntropyLoss(weight=class_w)

def predict(model, X, L, batch=256):
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), batch):
            out.append(model(X[i:i+batch].to(device), L[i:i+batch].to(device)).argmax(1).cpu())
    return torch.cat(out).numpy()

def scores(y_true, pred):
    return {"accuracy": accuracy_score(y_true, pred), "macro_f1": f1_score(y_true, pred, average="macro"),
            "weighted_f1": f1_score(y_true, pred, average="weighted"),
            "precision_macro": precision_score(y_true, pred, average="macro", zero_division=0),
            "recall_macro": recall_score(y_true, pred, average="macro", zero_division=0)}

def run(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    model = BiLSTMClassifier(VOCAB_SIZE).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loader = DataLoader(TensorDataset(Xtr, Ltr, ytr), batch_size=BATCH, shuffle=True)
    hist, best_f1, best_ep, wait, best_state = [], -1.0, 0, 0, None
    t0 = time.time()
    for ep in range(1, EPOCHS + 1):
        model.train(); run_loss = 0.0
        for xb, lb, yb in loader:
            xb, lb, yb = xb.to(device), lb.to(device), yb.to(device)
            opt.zero_grad()
            loss = criterion(model(xb, lb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            run_loss += loss.item() * len(xb)
        val_f1 = f1_score(yva.numpy(), predict(model, Xva, Lva), average="macro")
        hist.append((run_loss / len(train), val_f1))
        if val_f1 > best_f1:
            best_f1, best_ep, wait = val_f1, ep, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state)
    pred = predict(model, Xte, Lte)
    return {"seed": seed, "best_val_f1": best_f1, "best_epoch": best_ep, "epochs_run": len(hist),
            "train_time_s": time.time() - t0, "hist": hist, "pred": pred, **scores(yte.numpy(), pred)}

runs = []
for s in SEEDS:
    r = run(s)
    runs.append(r)
    print(f"seed {s}: best epoch {r['best_epoch']:2d}/{r['epochs_run']:2d}  val macro-F1 {r['best_val_f1']:.4f}  "
          f"test macro-F1 {r['macro_f1']:.4f}  test acc {r['accuracy']:.4f}  ({r['train_time_s']:.0f}s)")

## Step 5 - Mean and spread over seeds

The spread (standard deviation) tells you how much of a difference between two models could just be luck. If two models differ by less than their spread, you cannot say one is better.

In [ ]:
res = pd.DataFrame([{k: v for k, v in r.items() if k not in ("hist", "pred")} for r in runs])
res.to_csv("results/metrics/bilstm_runs.csv", index=False)
print(res.round(4).to_string(index=False))
print()
for col in ["macro_f1", "accuracy", "weighted_f1"]:
    print(f"{col:12s} mean {res[col].mean():.4f}   std {res[col].std(ddof=1):.4f}   min {res[col].min():.4f}   max {res[col].max():.4f}")

## Step 6 - Training curve (the best-validation run)

In [ ]:
import matplotlib.pyplot as plt

rep = runs[int(np.argmax([r["best_val_f1"] for r in runs]))]
print("Representative run (best validation score): seed", rep["seed"])

fig, ax1 = plt.subplots(figsize=(7.5, 4.2))
ep = range(1, len(rep["hist"]) + 1)
ax1.plot(ep, [h[0] for h in rep["hist"]], color="tab:red", marker="o", ms=3)
ax1.set_xlabel("epoch"); ax1.set_ylabel("train loss", color="tab:red")
ax2 = ax1.twinx()
ax2.plot(ep, [h[1] for h in rep["hist"]], color="tab:blue", marker="s", ms=3)
ax2.set_ylabel("validation macro-F1", color="tab:blue")
ax2.axvline(rep["best_epoch"], color="gray", linestyle="--", alpha=0.6)
plt.title(f"BiLSTM training curve (seed {rep['seed']})")
plt.tight_layout()
plt.savefig("figures/bilstm/fig_training_curve.png", dpi=150, bbox_inches="tight")
plt.show()

## Step 7 - Per-class report and confusion matrix (representative run)

These are for the best-validation run only; the headline numbers are the means from Step 5.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

y_true, pred = yte.numpy(), rep["pred"]
print(classification_report(y_true, pred, target_names=LABELS, zero_division=0))

cm = confusion_matrix(y_true, pred, normalize="true")
fig, ax = plt.subplots(figsize=(5.5, 4.8))
ConfusionMatrixDisplay(cm, display_labels=LABELS).plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
ax.set_title("BiLSTM: normalised confusion matrix (test)")
plt.tight_layout()
plt.savefig("results/confusion_matrices/cm_bilstm.png", dpi=150, bbox_inches="tight")
plt.show()
print(pd.DataFrame(confusion_matrix(y_true, pred), index=[f"true {l}" for l in LABELS], columns=[f"pred {l}" for l in LABELS]))

## Step 8 - Save metrics, predictions and errors

The model weights stay in the local `models/` folder only (they are small, but not needed in the repo).

In [ ]:
import os

row = {"model_name": "BiLSTM (mean of seeds)",
       **{k: round(res[k].mean(), 4) for k in ["accuracy", "macro_f1", "weighted_f1", "precision_macro", "recall_macro"]},
       "macro_f1_std": round(res["macro_f1"].std(ddof=1), 4), "train_time_s": round(res["train_time_s"].mean(), 1),
       "notes": f"{len(SEEDS)} seeds; emb100, hidden64 bi, max_len={MAX_LEN}, vocab={VOCAB_SIZE}, lr={LR}, batch={BATCH}"}
path = "results/metrics_summary.csv"
old = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
old = old[old.get("model_name", pd.Series(dtype=str)) != row["model_name"]] if len(old) else old
pd.concat([old, pd.DataFrame([row])], ignore_index=True).to_csv(path, index=False)
print(pd.read_csv(path)[["model_name", "accuracy", "macro_f1", "macro_f1_std", "weighted_f1"]])

pd.DataFrame({"id": test["id"], "true": y_true, "pred": pred}).to_csv("results/predictions/bilstm_test_preds.csv", index=False)
errs = test.assign(pred=pred)
errs = errs[errs["label_id"] != errs["pred"]].copy()
errs["pred_name"] = errs["pred"].map(lambda i: LABELS[i])
errs[["id", "text", "label", "pred_name"]].to_csv("results/errors/bilstm_errors.csv", index=False)
print("\nMisclassified in the representative run:", len(errs), "of", len(test))
print(errs.groupby(["label", "pred_name"]).size().sort_values(ascending=False))

## Last step - Save your work

Everything this notebook wrote is already in the project folder (`results/` and `figures/`). Save the notebook itself (Cmd+S).

## Done - notebook 3

Paste these back to me:
- Step 2: vocabulary size, MAX_LEN and the `<UNK>` shares
- Step 4: the five seed lines
- Step 5: the mean and std lines
- Step 7: the per-class report and confusion counts
- Step 8: the metrics table and the misclassification counts